# 11 — Country-only indices for an Insufficient-Data country (RESTRICTED)

Builds the full country-office deliverable — admin2 Excel, admin2 GeoPackage, admin1 summary — for a country that notebook 02 flags as **Insufficient Data** (`id_countries_insufficient_data`, e.g. Nicaragua). The global product keeps that country blank; this notebook does **not** change that.

**Restricted delivery.** Results are for the country office only — no global disclosure. Everything is written under `output_mali/restricted/`, never into `global/`, `results/` or `sdmx/`.

## What is (and is not) computed

- **Country scope only**: `cindex`, `cexpi_*`, `cexpc_*`, normalised across the country's own admin2 (and, in the summary, admin1) units.
- **No global / regional** indices or classes (`g*`, `r*`): the country was not part of the global/regional Fisher-Jenks run, so those columns are dropped rather than shipped blank. Country classes are **not comparable** with the global or regional classes.

## Steps

0. **Pixel counts** — NB00's exact reducer (same `define_hazard` masking, analysis scale, tileScale, duplicate twins), retrieved interactively (`getInfo`) instead of a Drive export since one country is small. Compared with the country's rows in the shared `hazs_count` table; those rows are replaced only if they differ (with a backup).
1. **Admin2 indices** — NB02 country block via `agg.compute_scope(..., "c", "adm0_ucode")` on the country partials from notebook 01. Population validated against `admin2_population.csv`.
2. **Excel** — as NB03 cell 9.
3. **GeoPackage** — NB03 cell 12, copied with paths redirected.
4. **Admin1 summary** — NB06 cells 2–3, copied with paths redirected and the country released from `agg.POP_BLANK_COUNTRIES` for this run only.

## Outputs (same shape as `results/{ucode}/`)

```
restricted/
  CSV/{hazard}_{PROD_DATE}.csv                     admin2 working tables (NB02 shape, no g*/r*)
  results/{ucode}/excel/{ucode}_{hazard}_{PROD_DATE}.xlsx
  results/{ucode}/geopackage/{ucode}_{hazard}_{PROD_DATE}.gpkg
  results/{ucode}/adm1_summary/...
  results/{ucode}/README.txt
```

## Execution order

Run after notebook 01 has produced the country partials. Independent of notebook 02.

In [ ]:
# ============================================================
# CCRI Hazard Statistics Processing in GEE.
# Country-only indices for Insufficient Data countries (restricted delivery).
# Administrative Level: ADM2 / ADM1.
# Author: Angelly Pugliese, Ph.D.
# Date: November 2026
# ============================================================

In [ ]:
# ============================================================
# Imports
# ============================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
LIB_DIR = PROJECT_ROOT / "lib"
CONFIG_DIR = PROJECT_ROOT / "config"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import os
import json
import shutil
import datetime as dt
import numpy as np
import pandas as pd
import geopandas as geopd
import shapely
import ee

from GEE_functions import GEEUtils, Hazard
from indicator_functions import IndicatorUtils
import aggregation_functions as agg

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 500)

In [ ]:
# ============================================================
# Parameters (edit these)
# ============================================================
COUNTRY_UCODE = "NIC_V1"
PROD_DATE = "2026-11-03"   # YYYY-MM-DD, stamped on every restricted output

OUTPUT_FOLDER = r"C:\Users\anjpu\Desktop\UNICEF\repo\CCRI\new_scripts\output_mali"
PARTIALS_FOLDER = f"{OUTPUT_FOLDER}/hazards"            # notebook 01 partials
RESTRICTED_ROOT = f"{OUTPUT_FOLDER}/restricted"
WORK_CSV = f"{RESTRICTED_ROOT}/CSV"

# Shared aux tables (same files NB05/NB06 use)
AUX_DIR = "output/aux"
POP_REFERENCE = f"{AUX_DIR}/admin2_population.csv"
COUNTS_TABLE = f"{AUX_DIR}/hazs_count_2026-09-11.csv"

# Step 0 (NB00) reduction settings: None -> country analysis scale, as NB00.
SCALE_OVERRIDE = None
TILE_SCALE = 4

# Countries still excluded from aggregation: everything NB06 excludes, except
# the country delivered here.
EXCLUDE_COUNTRIES = [c for c in agg.POP_BLANK_COUNTRIES if c != COUNTRY_UCODE]

# Hazard country code prefix of the admin2 ucodes, e.g. NIC_0001_0001_V1
ADM2_PREFIX = COUNTRY_UCODE.split("_")[0] + "_"


def restricted_path(path):
    # Every output of this notebook must land under RESTRICTED_ROOT.
    p = os.path.abspath(path)
    assert p.startswith(os.path.abspath(RESTRICTED_ROOT) + os.sep), f"refusing to write outside restricted/: {p}"
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return path


os.makedirs(WORK_CSV, exist_ok=True)

In [ ]:
# ============================================================
# Isolation snapshot: the published product must not change
# ============================================================
WATCHED = [f"{OUTPUT_FOLDER}/global", f"{OUTPUT_FOLDER}/results/{COUNTRY_UCODE}",
           f"{OUTPUT_FOLDER}/sdmx", POP_REFERENCE]


def snapshot(paths):
    snap = {}
    for root in paths:
        if os.path.isfile(root):
            snap[root] = os.stat(root).st_mtime_ns
            continue
        for d, _, files in os.walk(root):
            for f in files:
                fp = os.path.join(d, f)
                snap[fp] = os.stat(fp).st_mtime_ns
    return snap


snapshot_before = snapshot(WATCHED)
print(f"watching {len(snapshot_before):,} published files")

In [ ]:
# ============================================================
# Earth Engine + hazard metadata
# ============================================================
try:
    ee.Initialize(project="unicef-ccri")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="unicef-ccri")

unicef_data_source_path = "projects/unicef-ccri/assets"
utils = GEEUtils(ee)
iu = IndicatorUtils()

hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
hazard_list = [
    Hazard.from_dict(row, asset_prefix=unicef_data_source_path)
    for row in hazard_info.to_dict(orient="records")
]
codes = list(hazard_info["code"])
print(f"{COUNTRY_UCODE}: {len(hazard_list)} hazards")

## Step 0 — Pixel counts (`hazs_count`) for the country

NB00 cells 5–6 copied verbatim (helpers + scale rule). The reduction is the same as `submit_hazard_pixel_count_export`; only the sink differs: `getInfo()` instead of `Export.table.toDrive`.

In [ ]:
# ============================================================
# Build a non-blocking Export.table.toDrive task for one hazard x country.
# No getInfo() -> tasks run server-side in parallel instead of serially.
# ============================================================

# Hazards whose raw-image pixel count is identical to another hazard's, because
# they share the same asset/nodata/th_min and differ ONLY by threshold (which
# count() does not use). key = duplicate code, value = source code to copy from.
DUPLICATE_COUNT_OF = {
    "earthquake_pga_0p34": "earthquake_pga_0p09",
    "landslides_0p01":     "landslides_0p1",
}

DRIVE_FOLDER = "hazs_count_exports_fixed"   # single Drive folder for all task outputs

_ANALYSIS_SCALE_CACHE = {}   # country_ucode -> ee.Number | None

In [ ]:
def get_analysis_scale(country_ucode):
    """Return the country's population/analysis scale (WorldPop, ~92.77 m).

    This is the scale notebook 01 uses for the hazs_mean/std zonal reducers
    (GEEUtils.analysis_scale, set from the population raster in
    get_population_classes). Counting at the same scale is what makes
    hazs_count a valid weight for those means.

    Cached per country: the scale depends only on the country's population
    raster, not on the hazard, so it is resolved once per country instead of
    once per (country x hazard).

    Returns None for countries with no population raster; the caller then
    falls back to the hazard's true native scale.
    """
    if country_ucode in _ANALYSIS_SCALE_CACHE:
        return _ANALYSIS_SCALE_CACHE[country_ucode]

    if utils.check_country_no_population(country_ucode):
        scale = None
    else:
        adm2_fc = utils.get_admin2_boundaries(country_ucode, admin_level="adm0_ucode")
        utils.set_population_dicts(country_ucode, adm2_fc)
        utils.get_population_classes()      # sets utils.analysis_scale
        scale = utils.analysis_scale

    _ANALYSIS_SCALE_CACHE[country_ucode] = scale
    return scale


def get_native_scale(hazard):
    """Native scale of the hazard raster, safe for mosaicked assets.

    ImageCollection.mosaic() drops the projection, so nominalScale() on the
    mosaic returns the WGS84 default (111319.49 m). Take the scale from a
    member image instead. Only used as a fallback for no-population countries.
    """
    if hazard.mosaic:
        return ee.Image(ee.ImageCollection(hazard.asset).first()) \
                 .projection().nominalScale()
    return ee.Image(hazard.asset).projection().nominalScale()


def submit_hazard_pixel_count_export(
    hazard,
    country_ucode,
    scale_override=None,
    tile_scale=8,
    drive_folder=DRIVE_FOLDER,
):
    """Submit ONE non-blocking Export.table.toDrive task counting valid raw-hazard
    pixels per admin2 unit for `hazard` x `country_ucode`.

    Uses only ee.Reducer.count() over the raw (nodata-masked) hazard image — the
    same image domain define_hazard produces — so counts match the pixels that
    fed hazs_mean/std in notebook 01.

    Scale: the country's population/analysis scale, matching notebook 01's
    reducers. NOT the hazard image's nominalScale(), which is wrong (111 km)
    for the three mosaicked hazards. See the config cell.

    Returns the started ee.batch.Task, or None if skipped as a duplicate.
    Output CSV is named {country}__{hazard_code}.csv so results reconstruct
    per-country.
    """
    # Skip duplicates: their count is identical to the source hazard's.
    if hazard.code in DUPLICATE_COUNT_OF:
        return None

    images = utils.define_hazard(hazard)
    raw = images["hazard_raw_image"].rename("haz").toFloat()

    adm2_fc = utils.get_admin2_boundaries(country_ucode, admin_level="adm0_ucode")

    if scale_override:
        scale = scale_override
    else:
        # Match notebook 01's reduction scale; fall back to the hazard's true
        # native scale only where there is no population raster.
        scale = get_analysis_scale(country_ucode) or get_native_scale(hazard)

    reduced = raw.reduceRegions(
        collection=adm2_fc,
        reducer=ee.Reducer.count().setOutputs(["hazs_count"]),
        scale=scale,
        tileScale=tile_scale,
    ).select(["adm2_ucode", "hazs_count"], None, False)

    # Tag rows with the hazard code server-side so the CSV is self-describing.
    reduced = reduced.map(lambda f: f.set("hazard_code", hazard.code))

    desc = f"cnt_{country_ucode}_{hazard.code}"[:100]  # EE description limit
    fname = f"{country_ucode}__{hazard.code}"
    task = ee.batch.Export.table.toDrive(
        collection=reduced,
        description=desc,
        folder=drive_folder,
        fileNamePrefix=fname,
        fileFormat="CSV",
        selectors=["adm2_ucode", "hazard_code", "hazs_count"],
    )
    task.start()
    return task

In [ ]:
# ============================================================
# Retrieve the counts interactively (same reducer as NB00's export)
# ============================================================
def hazard_pixel_counts(hazard, country_ucode, scale_override=None, tile_scale=TILE_SCALE):
    images = utils.define_hazard(hazard)
    raw = images["hazard_raw_image"].rename("haz").toFloat()
    adm2_fc = utils.get_admin2_boundaries(country_ucode, admin_level="adm0_ucode")
    if scale_override:
        scale = scale_override
    else:
        scale = get_analysis_scale(country_ucode) or get_native_scale(hazard)
    reduced = raw.reduceRegions(
        collection=adm2_fc,
        reducer=ee.Reducer.count().setOutputs(["hazs_count"]),
        scale=scale,
        tileScale=tile_scale,
    ).select(["adm2_ucode", "hazs_count"], None, False)
    rows = [f["properties"] for f in reduced.getInfo()["features"]]
    out = pd.DataFrame(rows, columns=["adm2_ucode", "hazs_count"])
    out["hazard_code"] = hazard.code
    return out


frames = []
for hazard in hazard_list:
    if hazard.code in DUPLICATE_COUNT_OF:
        continue
    frames.append(hazard_pixel_counts(hazard, COUNTRY_UCODE, SCALE_OVERRIDE))
    print(f"  {hazard.code}: {len(frames[-1])} admin2")
new_counts = pd.concat(frames, ignore_index=True)

# Add duplicate hazards back in from their twins (NB00 cell 9).
for dup, src_code in DUPLICATE_COUNT_OF.items():
    twin = new_counts[new_counts.hazard_code == src_code].copy()
    twin["hazard_code"] = dup
    new_counts = pd.concat([new_counts, twin], ignore_index=True)
new_counts = new_counts[["adm2_ucode", "hazard_code", "hazs_count"]]
new_counts["hazs_count"] = pd.to_numeric(new_counts["hazs_count"], errors="coerce")
# Some boundary assets repeat an adm2 feature; keep one row per (adm2, hazard).
new_counts = new_counts.drop_duplicates(subset=["adm2_ucode", "hazard_code"], keep="first")
print(f"retrieved {len(new_counts)} rows ({new_counts.hazard_code.nunique()} hazards)")

In [ ]:
# ============================================================
# Compare with the shared table; replace this country's rows only if they differ
# ============================================================
counts_all = pd.read_csv(COUNTS_TABLE)
is_country = counts_all["adm2_ucode"].str.startswith(ADM2_PREFIX)
old_counts = counts_all[is_country].drop_duplicates(subset=["adm2_ucode", "hazard_code"])

cmp = old_counts.merge(new_counts, on=["adm2_ucode", "hazard_code"], how="outer",
                       suffixes=("_old", "_new"), indicator=True)
changed = cmp[(cmp["_merge"] != "both") | (cmp["hazs_count_old"] != cmp["hazs_count_new"])]
print(f"existing rows: {len(old_counts)} | retrieved: {len(new_counts)} | differing: {len(changed)}")

if changed.empty:
    print("hazs_count table already consistent for", COUNTRY_UCODE, "- left unchanged")
else:
    print(changed.groupby("hazard_code").size().to_string())
    backup = f"{COUNTS_TABLE}.bak-{dt.date.today().isoformat()}"
    shutil.copy2(COUNTS_TABLE, backup)
    updated = pd.concat([counts_all[~is_country], new_counts], ignore_index=True)
    updated.to_csv(COUNTS_TABLE, index=False)
    print(f"replaced {is_country.sum()} -> {len(new_counts)} {COUNTRY_UCODE} rows in {COUNTS_TABLE} (backup: {backup})")

counts_df = agg.load_counts_table(COUNTS_TABLE)

## Step 1 — Admin2 country-scope indices

Mirrors NB02 for one country without the Insufficient-Data blanking: `No Data` units are flagged as in NB02, and the country block (`cindex`, `cexpi_*`, `cexpc_*`) is computed with `agg.compute_scope`, which reuses `IndicatorUtils` exactly as NB02 does.

In [ ]:
# ============================================================
# Population check: hazard totals (not exposed) == admin2_population.csv
# ============================================================
POP_COLS = ["pop_total", "pop_total_female", "pop_total_male",
            "pop_under_18_female", "pop_under_18_male", "pop_under_18_total"]
pop_ref = pd.read_csv(POP_REFERENCE)
ref_c = pop_ref[pop_ref["adm2_ucode"].str.startswith(ADM2_PREFIX)].set_index("adm2_ucode")[POP_COLS].sort_index()
assert not ref_c.index.duplicated().any() and ref_c.notna().all().all()

for code_ in codes:
    p = pd.read_csv(f"{PARTIALS_FOLDER}/{code_}/partials/{COUNTRY_UCODE}_{code_}_admin2.csv")
    p = p.set_index("adm2_ucode")[POP_COLS].sort_index()
    assert p.index.equals(ref_c.index), f"{code_}: admin2 set differs from admin2_population.csv"
    diff = (p - ref_c).abs().max().max()
    assert diff == 0, f"{code_}: population differs from admin2_population.csv (max abs diff {diff})"
print(f"population OK: {len(ref_c)} admin2, identical across {len(codes)} hazards and admin2_population.csv")

In [ ]:
# ============================================================
# Country-scope indices per hazard (NB02 country block)
# ============================================================
POPULATION_CLASSES = agg.POPULATION_CLASSES
C_COLS = (["cindex"] + [f"cexpi_{pc}" for pc in POPULATION_CLASSES]
          + [f"cexpc_{pc}" for pc in POPULATION_CLASSES])
DISPLAY_COLS = iu.remove_non_country_level(iu.get_all_indicator_columns())


def build_country_admin2(partial_csv):
    df = pd.read_csv(partial_csv)
    # Some partials (e.g. landslides_0p1) carry both adm2_name/adm2_ucode and the
    # legacy name/ucode, which the rename map would turn into duplicate columns.
    # Keep one, filled from whichever copy is populated (as agg.load_hazard_global).
    for legacy, base in (("name", "adm2_name"), ("ucode", "adm2_ucode")):
        if legacy in df.columns and base in df.columns:
            df[base] = df[base].where(df[base].notna(), df[legacy])
            df = df.drop(columns=[legacy])
    # Units without a name in the boundary asset would be dropped by NB06's
    # pivot on (adm1_ucode, adm1_name); give them a visible placeholder.
    df["adm1_name"] = df["adm1_name"].fillna("Unnamed (" + df["adm1_ucode"] + ")")
    df["status"] = ""
    mask = df["hazs_mean"].isnull() & df["exp_total"].isnull()
    df.loc[mask, "status"] = "No Data"

    # NB02 initialises cexpi to 0 and cindex/cexpc to NaN for every row.
    df = df.assign(cindex=np.nan, **{f"cexpi_{pc}": 0.0 for pc in POPULATION_CLASSES},
                   **{f"cexpc_{pc}": np.nan for pc in POPULATION_CLASSES})
    ok = df["status"] == ""
    # <= 2 units would borrow the global class, which does not exist here.
    assert ok.sum() > 2, f"only {ok.sum()} usable admin2 units"
    scoped = agg.compute_scope(iu, df.loc[ok], "c", "adm0_ucode")
    df.loc[ok, C_COLS] = scoped[C_COLS].values

    df = df.rename(columns=iu.get_rename_map())
    df = iu.round_cols(df)
    return df[DISPLAY_COLS]


for code_ in codes:
    out = build_country_admin2(f"{PARTIALS_FOLDER}/{code_}/partials/{COUNTRY_UCODE}_{code_}_admin2.csv")
    out.to_csv(restricted_path(f"{WORK_CSV}/{code_}_{PROD_DATE}.csv"), index=False)
    n_nd = (out["status"] == "No Data").sum()
    print(f"  {code_:38s} {len(out)} admin2, {n_nd} No Data, cexpc classes {sorted(out['cexpc'].dropna().unique().astype(int))}")

In [ ]:
# ============================================================
# Method parity: the same function on a published country reproduces NB02
# ============================================================
PARITY_UCODE = "HND_V1"
PUBLISHED_DATE = "2026-08-31"
c_cols_renamed = [iu.get_rename_map().get(c, c) for c in C_COLS]

worst = 0.0
for code_ in codes:
    mine = build_country_admin2(f"{PARTIALS_FOLDER}/{code_}/partials/{PARITY_UCODE}_{code_}_admin2.csv")
    pub = agg.load_hazard_global(code_, PUBLISHED_DATE, f"{OUTPUT_FOLDER}/global/CSV")
    pub = pub[pub["adm0_ucode"] == PARITY_UCODE]
    m = mine.set_index("adm2_ucode")[c_cols_renamed].sort_index()
    p = pub.set_index("adm2_ucode")[c_cols_renamed].sort_index()
    assert m.index.equals(p.index), f"{code_}: admin2 set differs"
    d = (m.astype(float) - p.astype(float)).abs()
    both_nan = m.isna() & p.isna()
    one_nan = m.isna() ^ p.isna()
    assert not one_nan.any().any(), f"{code_}: NaN pattern differs"
    worst = max(worst, d[~both_nan].max().max())
print(f"parity vs published {PARITY_UCODE}: max abs difference over {len(c_cols_renamed)} country columns x {len(codes)} hazards = {worst}")
assert worst < 1e-9, "country-scope computation does not reproduce NB02"

## Step 2 — Excel (NB03 cell 9)

In [ ]:
# NB03 reads the CSV with index_col=0, so the delivered Excel starts at `hazard`
# (geo_level becomes the index and is dropped). Kept identical here.
restricted_path(f"{RESTRICTED_ROOT}/results/{COUNTRY_UCODE}/excel/x")
for code_ in codes:
    df_country = pd.read_csv(f"{WORK_CSV}/{code_}_{PROD_DATE}.csv", index_col=0)
    df_country.reset_index(drop=True, inplace=True)
    utils.save_country_results_to_excel(df_country, code_, COUNTRY_UCODE, RESTRICTED_ROOT, date=PROD_DATE)

## Step 3 — GeoPackage (NB03 cells 11–12, paths redirected)

In [ ]:
def polygons_only(geom:shapely.Geometry):
    # If it is not a collection, just return it as-is
    if geom.geom_type != "GeometryCollection":
        return geom

    # Filter only Polygon / MultiPolygon parts
    polys = []
    for g in geom.geoms:  # GeometryCollection exposes its members via .geoms [web:1]
        if g.geom_type == "Polygon":
            polys.append(g)
        elif g.geom_type == "MultiPolygon":
            polys.extend(list(g.geoms))  # flatten nested MultiPolygons [web:10]

    if not polys:
        return None  # or geom, or an empty GeometryCollection, depending on your needs

    if len(polys) == 1:
        return polys[0]  # single Polygon
    else:
        return shapely.geometry.MultiPolygon(polys)  # build a MultiPolygon from a list of Polygons 

In [ ]:
# Export as Geopackage (GPKG)
# Resumable: any {country, hazard} whose .gpkg already exists for prod_date is
# skipped, so this cell can be re-run after a crash and picks up where it stopped.
adm0_ucode_col = "adm0_ucode"

# Set to True to ignore existing files and rebuild everything from scratch.
FORCE_REBUILD_GPKG = True

prod_date = PROD_DATE
countries_to_process = [COUNTRY_UCODE]
PROCESS_SPECIFIC_HAZARDS = None

hazards_this_run = [
    h for h in hazard_list
    if not (PROCESS_SPECIFIC_HAZARDS and h.code not in PROCESS_SPECIFIC_HAZARDS)
]

n_skipped = 0
n_written = 0
failed_countries = []

for country_ucode in countries_to_process:
    country_folder_path = f"{RESTRICTED_ROOT}/results/{country_ucode}/geopackage"
    restricted_path(f"{country_folder_path}/x")

    # Which hazards still need writing for this country?
    pending_hazards = []
    for hazard in hazards_this_run:
        out_gpkg = f"{country_folder_path}/{country_ucode}_{hazard.code}_{prod_date}.gpkg"
        if not FORCE_REBUILD_GPKG and os.path.exists(out_gpkg):
            n_skipped += 1
        else:
            pending_hazards.append(hazard)

    # Country fully done -> skip entirely, no GEE geometry fetch.
    if not pending_hazards:
        print(f"[skip] {country_ucode}: all {len(hazards_this_run)} hazards already exported")
        continue

    print(f"[work] {country_ucode}: {len(pending_hazards)} hazard(s) pending")

    # for each country a new adm2 dataset is built
    all_adm2_country = None

    try:
        for hazard in pending_hazards:
            hazard_code = hazard.code
            hazard_name: str = hazard.name
            print(f"Starting Geopackage save for: {hazard_name} - {country_ucode}")
            # import the global one
            df_global = pd.read_csv(f"{WORK_CSV}/{hazard_code}_{prod_date}.csv", index_col=0)
            # get the country-specific data
            df_country: pd.DataFrame = df_global.loc[df_global[adm0_ucode_col] == country_ucode]
            df_country.reset_index(drop=True, inplace=True)

            # First time, fetch the geometries from GEE
            if all_adm2_country is None:
                print(f"extracting adm2 features for {country_ucode}")
                # build geometries for each adm2 in the country
                adm2_gee_asset = ee.FeatureCollection(f'{unicef_data_source_path}/{utils.ADMIN2_ASSET}')
                adm1_list = list(df_country["adm1_ucode"].unique())

                # get all adm2 geometries by fetching by adm1
                all_adm2_list = []
                for adm1 in adm1_list:
                    # .distinct() server-side: the asset repeats some adm2 units as
                    # separate features with identical geometry/properties, so ask GEE
                    # for one feature per adm2_ucode rather than filtering after the fetch.
                    features = (
                        adm2_gee_asset
                        .filter(ee.Filter.eq("adm1_ucode", adm1))
                        .distinct("adm2_ucode")
                        .getInfo()["features"]
                    )
                    partial_adm2 = []
                    for f in features:
                        geometry = shapely.geometry.shape(f["geometry"])
                        geometry = polygons_only(geometry)

                        formatted_feature = {"geometry": geometry}
                        formatted_feature.update(f["properties"])
                        partial_adm2.append(formatted_feature)

                    all_adm2_list += partial_adm2

                all_adm2_country = pd.DataFrame(all_adm2_list)

                # Safety net only: .distinct() above already deduplicates server-side.
                # This catches anything that slips through (e.g. if the asset later
                # repeats a ucode with differing properties, which distinct() keeps).
                n_before = len(all_adm2_country)
                all_adm2_country = all_adm2_country.drop_duplicates(subset="adm2_ucode", keep="first")
                if len(all_adm2_country) != n_before:
                    print(f"  note: dropped {n_before - len(all_adm2_country)} duplicate adm2 record(s) "
                          f"for {country_ucode} ({n_before} -> {len(all_adm2_country)})")

            # Join explicitly on adm2_ucode. The previous implicit merge joined on
            # every shared column, which breaks when a shared key differs in dtype
            # between the CSV and GEE (e.g. MLI adm0_id) and can silently drop rows.
            cols = list(df_country.columns) + ["geometry"]
            df_country = pd.merge(
                df_country,
                all_adm2_country[["adm2_ucode", "geometry"]],
                on="adm2_ucode",
                how="right",
            )
            df_country = df_country[cols]
            geodf_country = geopd.GeoDataFrame(df_country, geometry="geometry")
            geodf_country.set_crs(epsg=4326, inplace=True)

            # create folder if it does not exist
            os.makedirs(country_folder_path, exist_ok=True)

            # Write to a temp path first, then rename, so an interrupted write
            # never leaves a truncated .gpkg that a later resume would treat as done.
            out_gpkg = f"{country_folder_path}/{country_ucode}_{hazard_code}_{prod_date}.gpkg"
            tmp_gpkg = out_gpkg + ".tmp"
            if os.path.exists(tmp_gpkg):
                os.remove(tmp_gpkg)
            geodf_country.to_file(tmp_gpkg, index=False, driver="GPKG", layer=hazard_code)
            os.replace(tmp_gpkg, out_gpkg)

            n_written += 1
            print(f"Exported {hazard_code} results for country {country_ucode}: {out_gpkg}")
            print()
    except Exception as e:
        # Log and move to the next country instead of killing the whole run.
        failed_countries.append((country_ucode, str(e)))
        print(f"ERROR on {country_ucode}: {e}")
        print()

print("=" * 60)
print(f"Done. Written: {n_written}   Skipped (already existed): {n_skipped}")
if failed_countries:
    print(f"Failed countries ({len(failed_countries)}):")
    for uc, err in failed_countries:
        print(f"  {uc}: {err}")
else:
    print("No country-level failures.")

## Step 4 — Admin1 summary (NB06 cells 2–3)

Copied verbatim except: parameters point to the restricted working CSVs and outputs; `aggregate_counts` and the conservation check use `EXCLUDE_COUNTRIES` (NB06's list minus this country), the same mechanism NB09 uses for Kosovo.

In [ ]:
# ============================================================
# Parameters (edit these)
# ============================================================
# List of country ucodes to summarize. Set to None to process every country
# found in the source CSVs.
COUNTRY_UCODES = [COUNTRY_UCODE]
ADM2_PROD_DATE = PROD_DATE
SOURCE_FOLDER = WORK_CSV
# Independent admin2 population source. Population is a property of the PLACE,
# not of the hazard: the per-hazard tables blank it for suppressed units, so
# reading it from them makes the denominator drift per hazard. Sourcing it here
# is what keeps every hazard reporting identical totals (same approach NB05
# already uses for the SDMX population stream).

POP = agg.POPULATION_CLASSES

# Which per-hazard values to carry into the WIDE prioritization file
# Children-focused hazard_exposure wide view: exposure class + index use the
# under-18 columns; cindex (hazard magnitude) is population-independent.
WIDE_VALUES = ["cexpi_under_18_total", "cexpc_under_18_total", "cindex"]


hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
short_code = {r["code"]: r["short_code"] for _, r in hazard_info.iterrows()}
counts_df = agg.load_counts_table(COUNTS_TABLE)
print("weight table:", "loaded" if counts_df is not None else "MISSING -> pop_total fallback")

pop_ref = pd.read_csv(POP_REFERENCE) if os.path.exists(POP_REFERENCE) else None
print("population reference:", f"loaded ({len(pop_ref):,} admin2)" if pop_ref is not None
      else "MISSING -> population will be read from the hazard tables (drifts per hazard!)")


# Discover hazard CSVs in the source folder, then keep only hazards that are
# still defined in hazard_info.json. This drops retired hazards (e.g. the old
# non-grid complex_emergencies_acled) even if a stale CSV lingers in the folder.
valid_codes = set(hazard_info["code"])
discovered = agg.discover_hazard_codes(SOURCE_FOLDER, ADM2_PROD_DATE)
codes = [c for c in discovered if c in valid_codes]
dropped = [c for c in discovered if c not in valid_codes]
if dropped:
    print(f"skipping {len(dropped)} hazard(s) not in hazard_info.json: {dropped}")
print(f"hazards: {len(codes)}")


# ============================================================
# Which countries can actually be summarized to admin1?
# ============================================================
# Two structural cases are excluded:
#  * 1:1 admin1->admin2 -- aggregation is a pass-through, so the admin1 summary
#    would just restate the admin2 data at the same granularity.
#  * a single admin1 unit -- there are no peers to normalize against; these are
#    also a subset of the 1:1 case whenever the country has one admin2.
_probe = agg.load_hazard_global(codes[0], ADM2_PROD_DATE, SOURCE_FOLDER)
_structure = (_probe.groupby("adm0_ucode")
                    .agg(n_adm1=("adm1_ucode", "nunique"),
                         n_adm2=("adm2_ucode", "nunique")))

_requested = COUNTRY_UCODES if COUNTRY_UCODES is not None else sorted(_structure.index)
_missing = [u for u in _requested if u not in _structure.index]

one_to_one = [u for u in _requested if u in _structure.index
              and _structure.loc[u, "n_adm1"] == _structure.loc[u, "n_adm2"]]
single_adm1 = [u for u in _requested if u in _structure.index
               and _structure.loc[u, "n_adm1"] == 1]
# single_adm1 with >1 admin2 is not 1:1, but still cannot be classified.
SKIP = sorted(set(one_to_one) | set(single_adm1))
COUNTRIES_TO_RUN = [u for u in _requested if u not in SKIP and u in _structure.index]

print(f"requested: {len(_requested)}")
if _missing:
    print(f"  not in source data ({len(_missing)}): {_missing[:10]}")
print(f"  skipped 1:1 admin1==admin2 : {len(one_to_one)}")
print(f"  skipped single admin1 unit : {len(single_adm1)}")
print(f"  -> summarizing {len(COUNTRIES_TO_RUN)} country(ies)")

In [ ]:
# ============================================================
# Main loop: one country at a time
# ============================================================
# Cells 3-8 of the original single-country notebook are combined here so the
# whole pipeline (aggregate -> long -> wide -> two Excel views -> verify) runs
# per country. The per-country logic inside the loop is unchanged.
summary_rows = []

for _n, COUNTRY_UCODE in enumerate(COUNTRIES_TO_RUN, 1):
    print(f'\n{"============================================================"}')
    print(f'[{_n}/{len(COUNTRIES_TO_RUN)}] {COUNTRY_UCODE}')

    out_dir = f'{RESTRICTED_ROOT}/results/{COUNTRY_UCODE}/adm1_summary'
    restricted_path(f'{out_dir}/x')
    os.makedirs(out_dir, exist_ok=True)

    try:
        # ============================================================
        # Per-hazard: filter country -> aggregate to admin1 -> country-scope indices
        # ============================================================
        long_rows = []
        for code in codes:
            df = agg.load_hazard_global(code, ADM2_PROD_DATE, SOURCE_FOLDER)
            c = df[df["adm0_ucode"] == COUNTRY_UCODE].copy()
            if c.empty:
                print(f"  {code}: no rows for {COUNTRY_UCODE}, skipped")
                continue

            # Population from the independent reference BEFORE aggregating, so
            # suppressed units still contribute their people to the denominator.
            if pop_ref is not None:
                c = agg.apply_population_reference(c, pop_ref)
            c = agg.weight_table_join(c, counts_df, code)
            c = agg.recompute_exposure_inputs(c)
            a1 = agg.aggregate_counts(c, "adm1_ucode", exclude_countries=EXCLUDE_COUNTRIES)
            if a1.empty:
                print(f"  {code}: no usable admin1 units (all flagged), skipped")
                continue
            a1 = agg.recompute_exposure_inputs(a1)

            # Country-scope indices/classes, normalized across this country's admin1 units
            a1 = iu.compute_hazard_index(a1, hazs_max_col="hazs_max", output_col="cindex")
            for pc in POP:
                a1 = iu.compute_exposure_index(a1, f"{pc}_log_exposure", f"{pc}_relative_exposure", output_col=f"cexpi_{pc}")
                n_units = len(a1)
                if n_units < 2:
                    # A single admin1 unit has no peers to rank against, so a
                    # country-scope class is undefined. The global classification
                    # leaves these empty (NaN) rather than 0 -- 0 already means
                    # "no exposure" (aexp == 0) -- so match that convention here.
                    a1[f"cexpc_{pc}"] = np.nan
                elif a1[f"cexpi_{pc}"].max() == 0:
                    a1[f"cexpc_{pc}"] = 0
                else:
                    k = min(5, max(3, n_units))
                    a1 = iu.compute_exposure_class(a1, f"cexpi_{pc}", f"{pc}_log_exposure", k=k, output_col=f"cexpc_{pc}")

            a1["hazard_code"] = code
            a1["hazard_short"] = short_code.get(code, code)
            long_rows.append(a1)
            print(f"  {code}: {len(a1)} admin1 units")

        if not long_rows:
            # Every admin2 row for this country is flagged (Insufficient Data /
            # Not Applicable), so aggregate_counts drops them all and there is
            # nothing to summarize. Report it plainly instead of letting
            # pd.concat raise "No objects to concatenate".
            raise RuntimeError("no usable admin2 rows (all flagged) for any hazard")

        long = pd.concat(long_rows, ignore_index=True)

        # `n_children` (from aggregation_functions.aggregate_counts) counts the admin2
        # units aggregated into each admin1 -- NOT the under-18 population. Renamed to
        # `n_admin2` to remove that ambiguity in this child-hazard dataset.
        long = long.rename(columns={"n_children": "n_admin2"})

        print(f"\nLONG: {long.shape}, hazards={long.hazard_code.nunique()}, admin1={long.adm1_ucode.nunique()}")

        # ============================================================
        # LONG output (tidy: one row per admin1 x hazard)
        # ============================================================
        rel_cols = ["total_relative_exposure", "under_18_total_relative_exposure"]
        cexpi_cols = [f"cexpi_{pc}" for pc in POP]
        cexpc_cols = [f"cexpc_{pc}" for pc in POP]
        # hazard_short is dropped from the long output (hazard_code is the full identifier);
        # it is still used internally below to build the WIDE pivot columns.
        long_cols = (
            ["hazard_code", "iso3", "adm0_ucode", "adm1_ucode", "adm1_name",
             "unicef_region", "n_admin2", "hazs_mean", "hazs_std", "hazs_min", "hazs_max"]
            + agg.COUNT_COLS + rel_cols + ["cindex"] + cexpi_cols + cexpc_cols + ["status"]
        )
        # Coverage diagnostics travel with the long output so a consumer can
        # judge whether hazs_mean/std/min/max are based on enough of the unit.
        # Covered/total is TWO numeric columns, never a "3/7" string: Excel
        # autoconverts that to a date on open ("5/12" -> 05-Dec).
        # Coverage as two numeric counts, never a "3/7" string (Excel turns
        # that into a date: "5/12" -> 05-Dec). n_admin2_covered is inserted
        # directly after n_admin2 so the pair is always read together.
        if "n_admin2_covered" in long.columns and "n_admin2_covered" not in long_cols:
            if "n_admin2" in long_cols:
                long_cols.insert(long_cols.index("n_admin2") + 1, "n_admin2_covered")
            else:
                long_cols = long_cols + ["n_admin2_covered"]
        long_out = long[[c for c in long_cols if c in long.columns]].copy()
        # Explicit, stable ordering: hazard then adm1_ucode (pcode). The admin1
        # sequence within each hazard block then matches both workbooks exactly.
        long_out = long_out.sort_values(["hazard_code", "adm1_ucode"])
        long_out = iu.round_cols(long_out)
        long_path = f"{out_dir}/{COUNTRY_UCODE}_admin1_long_{PROD_DATE}.csv"
        long_out.to_csv(long_path, index=False)
        print(f"wrote {long_path}  ({long_out.shape})")

        # ============================================================
        # WIDE output (one row per admin1; per-hazard column blocks) for prioritization
        # ============================================================
        wide = long.pivot_table(
            index=["adm1_ucode", "adm1_name"],
            columns="hazard_short",
            values=WIDE_VALUES,
            aggfunc="first",
        )
        wide.columns = [f"{sc}_{val}" for val, sc in wide.columns]
        wide = wide.reset_index()

        # Cross-hazard prioritization helpers: how many hazards at each class, worst class, mean index
        class_cols = [c for c in wide.columns if c.endswith("_cexpc_under_18_total")]
        index_cols = [c for c in wide.columns if c.endswith("_cexpi_under_18_total")]
        wide["max_cexpc"] = wide[class_cols].max(axis=1)
        wide["n_hazards_class5"] = (wide[class_cols] == 5).sum(axis=1)
        wide["n_hazards_class4plus"] = (wide[class_cols] >= 4).sum(axis=1)
        wide["mean_cexpi"] = wide[index_cols].mean(axis=1).round(1)
        # Ordered by adm1_ucode (pcode), NOT by priority: COs read the long CSV
        # and both workbooks side by side, so every output must present the
        # admin1 units in the same order. The priority columns (max_cexpc,
        # n_hazards_class5, n_hazards_class4plus, mean_cexpi) are still in the
        # sheet, so a reader can re-sort on them in Excel whenever they want
        # the ranked view.
        wide = wide.sort_values("adm1_ucode")

        # The wide table is not written as CSV -- it is delivered as the colour-coded
        # Excel files built in the cells below, which use this same `wide` DataFrame.
        print(f"wide table built in memory  ({wide.shape})")

        # ============================================================
        # WIDE output as a COLOR-CODED Excel (prioritization view)
        # Columns are grouped BY HAZARD (Class | Exposure Index | Hazard Index), with a
        # merged two-row header: hazard display name on top, variable label below.
        # The exposure-CLASS column of each hazard is filled with the official CCRI
        # palette (config/class_palette.json); text flips to white on dark fills.
        # Labels come from hazard_info.json `display_name` and config/column_labels.json.
        # ============================================================
        import json
        from openpyxl import Workbook
        from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

        PALETTE = json.loads((CONFIG_DIR / "class_palette.json").read_text(encoding="utf-8"))
        PAL_CATS, CAT_ALIAS, CLASS_LABELS = PALETTE["categories"], PALETTE["hazard_info_category_alias"], PALETTE["class_labels"]
        LABELS = json.loads((CONFIG_DIR / "column_labels.json").read_text(encoding="utf-8"))
        VAR_LABELS, SUMMARY_LABELS, ID_LABELS = LABELS["variables"], LABELS["summary_columns"], LABELS["id_columns"]

        short_disp = {r["short_code"]: r["display_name"] for _, r in hazard_info.iterrows()}
        short_cat = {r["short_code"]: CAT_ALIAS.get(r["category"], r["category"]) for _, r in hazard_info.iterrows()}

        # Per-hazard column order within each group. Matches WIDE_VALUES.
        GROUP_ORDER = ["cexpc_under_18_total", "cexpi_under_18_total", "cindex"]  # Children: Class, Exposure Index, Hazard Index

        # Hazards present in `wide`, in hazard_info order (then any extras, e.g. conflict full-code).
        present = [r["short_code"] for _, r in hazard_info.iterrows()
                   if any(f"{r['short_code']}_{v}" in wide.columns for v in GROUP_ORDER)]
        known = set(present)
        for col in wide.columns:
            for v in GROUP_ORDER:
                if col.endswith(f"_{v}"):
                    sc = col[:-len(f"_{v}")]
                    if sc not in known:
                        present.append(sc); known.add(sc)

        id_cols = ["adm1_ucode", "adm1_name"]
        summary_cols = [c for c in ["max_cexpc", "n_hazards_class5", "n_hazards_class4plus", "mean_cexpi"] if c in wide.columns]
        group_spans = [(sc, [f"{sc}_{v}" for v in GROUP_ORDER if f"{sc}_{v}" in wide.columns]) for sc in present]
        group_spans = [(sc, cols) for sc, cols in group_spans if cols]
        final_cols = id_cols + [c for _, cols in group_spans for c in cols] + summary_cols
        wide2 = wide[final_cols].copy()


        def _class_int(v):
            n = pd.to_numeric(v, errors="coerce")
            if pd.isna(n):
                return None
            n = int(n)
            return n if 1 <= n <= 5 else None


        def _fill(cat, v):
            n = _class_int(v)
            if n is None:
                return None, None
            hx = PAL_CATS.get(cat, {}).get(CLASS_LABELS[n - 1])
            return (PatternFill("solid", fgColor="FF" + hx.lstrip("#").upper()), hx) if hx else (None, None)


        def _white(hx):
            h = hx.lstrip("#"); r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))
            return 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.5


        wb = Workbook()
        ws = wb.active
        ws.title = "prioritization"
        H1, H2, DATA0 = 1, 2, 3  # header row 1 (hazard), header row 2 (variable), first data row

        thin = Side(style="thin", color="DDDDDD")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)
        hdr_fill = PatternFill("solid", fgColor="FF305496")
        hdr_font = Font(color="FFFFFF", bold=True)
        center = Alignment(horizontal="center", vertical="center", wrap_text=True)

        col = 1
        for name in id_cols:  # id columns: label spans both header rows
            ws.merge_cells(start_row=H1, start_column=col, end_row=H2, end_column=col)
            c = ws.cell(H1, col, ID_LABELS.get(name, name)); c.fill = hdr_fill; c.font = hdr_font; c.alignment = center
            col += 1
        for sc, cols in group_spans:  # hazard groups: merged name on row1, variable on row2
            start = col
            for cn in cols:
                var = cn[len(sc) + 1:]
                c2 = ws.cell(H2, col, VAR_LABELS.get(var, var)); c2.fill = hdr_fill; c2.font = hdr_font; c2.alignment = center; c2.border = border
                col += 1
            ws.merge_cells(start_row=H1, start_column=start, end_row=H1, end_column=col - 1)
            c1 = ws.cell(H1, start, short_disp.get(sc, sc)); c1.fill = hdr_fill; c1.font = hdr_font; c1.alignment = center; c1.border = border
        for name in summary_cols:
            ws.merge_cells(start_row=H1, start_column=col, end_row=H2, end_column=col)
            c = ws.cell(H1, col, SUMMARY_LABELS.get(name, name)); c.fill = PatternFill("solid", fgColor="FF7F7F7F"); c.font = hdr_font; c.alignment = center
            col += 1

        for i, (_, row) in enumerate(wide2.iterrows()):  # data
            for j, name in enumerate(final_cols, start=1):
                ws.cell(DATA0 + i, j, row[name] if pd.notna(row[name]) else None)

        name_to_col = {name: j for j, name in enumerate(final_cols, start=1)}
        colored = 0
        for sc, _ in group_spans:  # color each hazard's class column
            cat = short_cat.get(sc); ccol = name_to_col.get(f"{sc}_cexpc_under_18_total")
            if cat is None or ccol is None:
                continue
            for i in range(len(wide2)):
                cell = ws.cell(DATA0 + i, ccol)
                fill, hx = _fill(cat, cell.value)
                if fill is None:
                    continue
                cell.fill = fill
                cell.font = Font(color="FFFFFF", bold=True) if _white(hx) else Font(bold=True)
                cell.alignment = Alignment(horizontal="center"); cell.border = border
                colored += 1

        ws.freeze_panes = "C3"  # below the 2-row header, right of the id columns
        xlsx_path = f"{out_dir}/{COUNTRY_UCODE}_admin1_wide_hazard_exposure_{PROD_DATE}.xlsx"
        wb.save(xlsx_path)
        print(f"wrote {xlsx_path}  ({len(group_spans)} hazard groups, {colored} class cells colored)")

        # ============================================================
        # EXPOSURE-focused wide Excel (second prioritization view)
        # Focuses on WHO is exposed rather than hazard intensity. Layout:
        #   front:  Admin1 Code | Admin1 Name | Total Population | Child Population
        #   per hazard (4 cols): Class (Total) | Class (Children) | Exposed (Total) | Exposed (Children)
        # Both class columns are colored with the CCRI palette. Population/exposed counts
        # use thousands formatting. Front population is the FULL admin1 total (max of
        # pop_total across hazards == the unmasked global-CSV admin1 population).
        # ============================================================
        from openpyxl import Workbook

        # Configs (reload so this cell is self-contained regardless of run order)
        _PAL = json.loads((CONFIG_DIR / "class_palette.json").read_text(encoding="utf-8"))
        PAL_CATS, CAT_ALIAS, CLASS_LABELS = _PAL["categories"], _PAL["hazard_info_category_alias"], _PAL["class_labels"]
        _LAB = json.loads((CONFIG_DIR / "column_labels.json").read_text(encoding="utf-8"))
        EXP_VARS, FRONT_LAB, ID_LAB = _LAB["exposure_view_variables"], _LAB["exposure_view_front_columns"], _LAB["id_columns"]

        code_disp = {r["code"]: r["display_name"] for _, r in hazard_info.iterrows()}
        code_cat = {r["code"]: CAT_ALIAS.get(r["category"], r["category"]) for _, r in hazard_info.iterrows()}

        EXP_VALUES = ["cexpc_total", "cexpc_under_18_total", "aexp", "u18_aexp"]
        CLASS_COLS = {"cexpc_total", "cexpc_under_18_total"}

        # hazards present in `long`, in hazard_info order (retired ones already filtered upstream)
        present = [c for c in hazard_info["code"] if c in set(long["hazard_code"])]

        # Full admin1 population = max of pop_total across hazards (== unmasked total).
        front = (long.groupby(["adm1_ucode", "adm1_name"], as_index=False)
                     .agg(pop_total=("pop_total", "max"), u18_pop_total=("u18_pop_total", "max")))

        # Pivot each exposure value into admin1 x hazard, then assemble hazard-grouped.
        wide_exp = front.set_index(["adm1_ucode", "adm1_name"])
        col_meta = []  # (final_col_name, hazard_code, value)
        for v in EXP_VALUES:
            _p = long.pivot_table(index=["adm1_ucode", "adm1_name"], columns="hazard_code", values=v, aggfunc="first")
            for code in present:
                if code in _p.columns:
                    wide_exp[f"{code}__{v}"] = _p[code]
        # reorder columns hazard-grouped
        for code in present:
            for v in EXP_VALUES:
                n = f"{code}__{v}"
                if n in wide_exp.columns:
                    col_meta.append((n, code, v))
        wide_exp = wide_exp[[n for n, _, _ in col_meta]].reset_index()
        wide_exp = front.merge(wide_exp, on=["adm1_ucode", "adm1_name"])

        # sort hotspots on top: worst total-population class, then total exposed
        class_total_cols = [n for (n, c, v) in col_meta if v == "cexpc_total"]
        # Same pcode ordering as the long CSV and the prioritization workbook.
        wide_exp = wide_exp.sort_values("adm1_ucode")


        def _class_int(v):
            n = pd.to_numeric(v, errors="coerce")
            if pd.isna(n):
                return None
            n = int(n)
            return n if 1 <= n <= 5 else None


        def _fill(cat, v):
            n = _class_int(v)
            if n is None:
                return None, None
            hx = PAL_CATS.get(cat, {}).get(CLASS_LABELS[n - 1])
            return (PatternFill("solid", fgColor="FF" + hx.lstrip("#").upper()), hx) if hx else (None, None)


        def _white(hx):
            h = hx.lstrip("#"); r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))
            return 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.5


        wb = Workbook()
        ws = wb.active
        ws.title = "exposure"
        H1, H2, DATA0 = 1, 2, 3

        hdr_fill = PatternFill("solid", fgColor="FF305496")
        front_fill = PatternFill("solid", fgColor="FF548235")
        hdr_font = Font(color="FFFFFF", bold=True)
        center = Alignment(horizontal="center", vertical="center", wrap_text=True)
        thin = Side(style="thin", color="DDDDDD")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)

        id_and_front = ["adm1_ucode", "adm1_name", "pop_total", "u18_pop_total"]
        final_cols = id_and_front + [n for (n, _, _) in col_meta]

        col = 1
        for name in id_and_front:
            ws.merge_cells(start_row=H1, start_column=col, end_row=H2, end_column=col)
            c = ws.cell(H1, col, ID_LAB.get(name) or FRONT_LAB.get(name) or name)
            c.fill = front_fill if name in FRONT_LAB else hdr_fill
            c.font = hdr_font; c.alignment = center
            col += 1
        for code in present:
            group = [(n, v) for (n, c, v) in col_meta if c == code]
            if not group:
                continue
            start = col
            for n, v in group:
                c2 = ws.cell(H2, col, EXP_VARS.get(v, v)); c2.fill = hdr_fill; c2.font = hdr_font; c2.alignment = center; c2.border = border
                col += 1
            ws.merge_cells(start_row=H1, start_column=start, end_row=H1, end_column=col - 1)
            c1 = ws.cell(H1, start, code_disp.get(code, code)); c1.fill = hdr_fill; c1.font = hdr_font; c1.alignment = center; c1.border = border

        for i, (_, row) in enumerate(wide_exp.iterrows()):
            for j, name in enumerate(final_cols, start=1):
                ws.cell(DATA0 + i, j, row[name] if pd.notna(row[name]) else None)

        name_to_col = {name: j for j, name in enumerate(final_cols, start=1)}
        colored = 0
        for (n, code, v) in col_meta:
            if v not in CLASS_COLS:
                continue
            cat = code_cat.get(code)
            if cat is None:
                continue
            ci = name_to_col[n]
            for i in range(len(wide_exp)):
                cell = ws.cell(DATA0 + i, ci)
                fill, hx = _fill(cat, cell.value)
                if fill is None:
                    continue
                cell.fill = fill
                cell.font = Font(color="FFFFFF", bold=True) if _white(hx) else Font(bold=True)
                cell.alignment = Alignment(horizontal="center"); cell.border = border
                colored += 1

        for name, j in name_to_col.items():  # thousands on population + exposed counts
            if name in ("pop_total", "u18_pop_total") or name.endswith("__aexp") or name.endswith("__u18_aexp"):
                for i in range(len(wide_exp)):
                    ws.cell(DATA0 + i, j).number_format = "#,##0"

        ws.freeze_panes = "C3"
        exp_path = f"{out_dir}/{COUNTRY_UCODE}_admin1_exposure_{PROD_DATE}.xlsx"
        wb.save(exp_path)
        print(f"wrote {exp_path}  ({len(present)} hazards, {colored} class cells colored)")

        # ============================================================
        # Verification
        # ============================================================
        # 1. POPULATION INVARIANCE (the check that would have caught the
        #    2026-09 denominator defect). Population must be identical for
        #    every hazard and must reconcile with the admin2 reference, for
        #    ALL SIX population classes -- not just pop_total.
        pop_ref_adm1 = None
        if pop_ref is not None:
            _b = agg.load_hazard_global(codes[0], ADM2_PROD_DATE, SOURCE_FOLDER)
            _b = _b[_b["adm0_ucode"] == COUNTRY_UCODE][["adm2_ucode", "adm1_ucode"]]
            pop_ref_adm1 = agg.apply_population_reference(
                _b.assign(**{c: np.nan for c in agg.COUNT_COLS}), pop_ref)

        problems = agg.validate_population_totals(long, pop_ref_adm1, "adm1_ucode")
        if problems.empty:
            n_cls = sum(1 for _, p in agg.CLASS_TO_COLS.values() if p in long.columns)
            print(f"population: OK -- invariant across {long.hazard_code.nunique()} "
                  f"hazards for all {n_cls} population classes, matches admin2 reference")
        else:
            print(f"population: {len(problems)} DISCREPANCIES")
            print(problems.head(12).to_string(index=False))
            raise AssertionError(
                f"{COUNTRY_UCODE}: population is not hazard-invariant "
                f"({len(problems)} discrepancies) -- refusing to publish")

        # 2. Conservation: summed admin1 aexp == summed admin2 aexp per hazard.
        #    Compared over ALL rows (not status=="") because No Data units now
        #    legitimately contribute 0 exposure rather than being dropped.
        for code in codes[:3]:
            df = agg.load_hazard_global(code, ADM2_PROD_DATE, SOURCE_FOLDER)
            c = df[(df["adm0_ucode"] == COUNTRY_UCODE)
                   & (~df["adm0_ucode"].isin(EXCLUDE_COUNTRIES))]
            adm2_sum = pd.to_numeric(c["aexp"], errors="coerce").fillna(0).sum()
            a1_sum = long.loc[long.hazard_code == code, "aexp"].sum()
            print(f"{code}: adm2 aexp={adm2_sum:,.0f}  admin1 aexp={a1_sum:,.0f}  "
                  f"match={abs(adm2_sum - a1_sum) < 1}")

        # 3. Index/class ranges
        print("\ncexpi_total in [0,10]:", long["cexpi_total"].between(0, 10).all())
        print("cexpc_total in {0..5}:", set(long["cexpc_total"].dropna().unique()).issubset({0, 1, 2, 3, 4, 5}))
        print("admin1 count == distinct adm1_ucode:", long.adm1_ucode.nunique())

        # 4. Coverage summary: where are the hazard statistics thin?
        if "n_admin2_covered" in long.columns:
            thin = long[(long.n_admin2_covered < 0.5 * long.n_admin2)
                        & long.hazs_mean.notna()]
            print(f"thin-coverage rows (<half the admin2 units have data, "
                  f"hazs_mean still reported): {len(thin)} of {len(long)}")

        summary_rows.append({'adm0_ucode': COUNTRY_UCODE,
                             'admin1_units': int(long.adm1_ucode.nunique()),
                             'hazards': int(long.hazard_code.nunique()),
                             'status': 'ok'})
    except Exception as _e:
        print(f'  ERROR {type(_e).__name__}: {_e}')
        summary_rows.append({'adm0_ucode': COUNTRY_UCODE, 'admin1_units': 0,
                             'hazards': 0, 'status': f'{type(_e).__name__}: {_e}'})

run_summary = pd.DataFrame(summary_rows)
print()
print('ok=', (run_summary.status == 'ok').sum(),
      ' failed=', (run_summary.status != 'ok').sum())
if (run_summary.status != 'ok').any():
    print(run_summary[run_summary.status != 'ok'].to_string(index=False))

In [ ]:
assert (run_summary.status == "ok").all(), "admin1 summary failed"

## README and final checks

In [ ]:
readme = f'''{COUNTRY_UCODE} - hazard exposure results (RESTRICTED: country office use only)
Produced {PROD_DATE} by notebook 11_country_only_indices_restricted.

NOT FOR GLOBAL DISCLOSURE. In the global CCRI/GCHD product this country is flagged
"Insufficient Data" and its values are blank. These files are for internal use by
the country office.

What is included
- excel/        one workbook per hazard, one row per admin2 unit
- geopackage/   the same tables with admin2 geometry (layer = hazard code)
- adm1_summary/ admin2 results aggregated to admin1 (long CSV + two colour-coded workbooks)

Indices and classes
- Only COUNTRY-scope indices are provided: cindex, cexpi/cmexpi/cfexpi (+u18_*),
  cexpc/cmexpc/cfexpc (+u18_*). They are normalised across this country's own units.
- Global and regional indices/classes (g*, r*) are NOT provided: the country was not
  part of the global/regional normalisation. Country classes must not be compared
  with the global or regional classes of other countries.
- status "No Data": the hazard layer does not cover the unit.

Sources
- Admin2 hazard statistics and exposure: notebook 01 partials ({PARTIALS_FOLDER}).
- Population: {POP_REFERENCE} (validated identical to the hazard tables).
- Pixel counts for admin1 mean/std weighting: {COUNTS_TABLE}.
'''
with open(restricted_path(f"{RESTRICTED_ROOT}/results/{COUNTRY_UCODE}/README.txt"), "w", encoding="utf-8") as f:
    f.write(readme)
print(readme)

In [ ]:
# ============================================================
# Final checks
# ============================================================
base = f"{RESTRICTED_ROOT}/results/{COUNTRY_UCODE}"
n_adm2 = len(ref_c)
for code_ in codes:
    x = pd.read_excel(f"{base}/excel/{COUNTRY_UCODE}_{code_}_{PROD_DATE}.xlsx")
    g = geopd.read_file(f"{base}/geopackage/{COUNTRY_UCODE}_{code_}_{PROD_DATE}.gpkg", layer=code_)
    assert len(x) == n_adm2 and len(g) == n_adm2, f"{code_}: rows excel={len(x)} gpkg={len(g)}"
    assert g.geometry.notna().all() and g["adm2_ucode"].notna().all(), f"{code_}: missing geometry/attributes"
    assert set(x.columns) <= set(DISPLAY_COLS), f"{code_}: unexpected (global/regional?) columns"
    ok = x["status"].fillna("") == ""
    assert x.loc[ok, "cindex"].between(0, 10).all() and set(x["cexpc"].dropna().unique()) <= {0, 1, 2, 3, 4, 5}
print(f"excel + geopackage OK: {len(codes)} hazards x {n_adm2} admin2")

from openpyxl import load_workbook
n_adm1 = long["adm1_ucode"].nunique()
for wb_name in ("admin1_wide_hazard_exposure", "admin1_exposure"):
    ws = load_workbook(f"{base}/adm1_summary/{COUNTRY_UCODE}_{wb_name}_{PROD_DATE}.xlsx", read_only=True).active
    n_rows = ws.max_row - 2   # two header rows
    assert n_rows == n_adm1, f"{wb_name}: {n_rows} admin1 rows, expected {n_adm1}"
print(f"admin1 workbooks OK: all {n_adm1} admin1 units present")

snapshot_after = snapshot(WATCHED)
changed_published = sorted(k for k in set(snapshot_before) | set(snapshot_after)
                           if snapshot_before.get(k) != snapshot_after.get(k))
assert not changed_published, f"published files changed: {changed_published[:5]}"
print("isolation OK: no file under global/, results/{0}/, sdmx/ or admin2_population.csv changed".format(COUNTRY_UCODE))